# Bike Rental Prediction in an Hour
This notebook audits the data, explores the main patterns, engineers useful features, trains a prediction model, and reports error metrics.

## 1. Import libraries and load the data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
sns.set_style('whitegrid')
df = pd.read_csv('hour.csv', parse_dates=['dteday'])
df.shape

## 2. Data audit
First I check missing values, duplicates, impossible ranges, and possible outliers so the analysis starts from clean data.

In [ ]:
display(df.head())
print('Missing values per column:')
display(df.isnull().sum())
print('Duplicate rows:', df.duplicated().sum())
range_checks = pd.DataFrame({
    'min': df[['season', 'hr', 'weathersit', 'temp', 'atemp', 'hum', 'windspeed', 'cnt']].min(),
    'max': df[['season', 'hr', 'weathersit', 'temp', 'atemp', 'hum', 'windspeed', 'cnt']].max()
})
display(range_checks)
def iqr_outliers(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return int(((series < lower) | (series > upper)).sum()), lower, upper
for col in ['temp', 'atemp', 'hum', 'windspeed', 'cnt']:
    n_out, lower, upper = iqr_outliers(df[col])
    print(f'{col}: outliers={n_out}, lower={lower:.3f}, upper={upper:.3f}')

## 3. Univariate analysis
I inspect the distribution of the numeric columns and the frequency of the categorical columns. For categorical features, I show both count and percentage so small groups are not hidden.

In [ ]:
numeric_cols = ['temp', 'atemp', 'hum', 'windspeed', 'cnt']
for col in numeric_cols:
    print(f'Column: {col} | count={df[col].notna().sum()} | skew={df[col].skew():.3f}')
    plt.figure(figsize=(6, 3))
    sns.histplot(df[col], bins=30, kde=True)
    plt.title(f'Distribution of {col}')
    plt.show()
df['cnt_log'] = np.log1p(df['cnt'])
print('cnt skew before:', round(df['cnt'].skew(), 3))
print('cnt_log skew after:', round(df['cnt_log'].skew(), 3))

In [ ]:
cat_cols = ['season', 'hr', 'weathersit']
for col in cat_cols:
    vc = df[col].value_counts().sort_index()
    pct = (df[col].value_counts(normalize=True).sort_index() * 100).round(2)
    summary = pd.DataFrame({'Count': vc, 'Percentage': pct})
    print(f'Column: {col}')
    display(summary)
    plt.figure(figsize=(8, 3))
    sns.countplot(x=col, data=df, order=vc.index)
    plt.title(f'Count Plot of {col}')
    plt.xlabel(col)
    plt.ylabel('Count')
    plt.show()

## 4. Interaction check
Before making causal claims, I look at how hour and season interact because demand at the same hour can change by season and weather.

In [ ]:
season_map = {1: 'spring', 2: 'summer', 3: 'fall', 4: 'winter'}
weather_map = {1: 'clear', 2: 'mist', 3: 'light_snow_rain', 4: 'heavy_rain_snow'}
df['season_name'] = df['season'].map(season_map)
df['weathersit_name'] = df['weathersit'].map(weather_map)
hour_season = df.pivot_table(index='hr', columns='season_name', values='cnt', aggfunc='mean')
plt.figure(figsize=(10, 4))
sns.heatmap(hour_season.T, cmap='YlGnBu')
plt.title('Average Rentals by Hour and Season')
plt.xlabel('Hour')
plt.ylabel('Season')
plt.show()
season_summary = df.groupby('season_name')['cnt'].agg(total_rentals='sum', avg_rentals='mean', observations='count')
display(season_summary)
plt.figure(figsize=(6, 4))
sns.barplot(x=season_summary.index, y=season_summary['avg_rentals'].values)
plt.title('Average Rentals by Season')
plt.xlabel('Season')
plt.ylabel('Average rentals')
plt.show()

## 5. Feature engineering
I create cyclic hour features, commute and weekend flags, and I remove leakage columns that reveal the target directly.

In [ ]:
df['hour_sin'] = np.sin(2 * np.pi * df['hr'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hr'] / 24)
df['is_commute'] = df['hr'].isin([7, 8, 17, 18]).astype(int)
df['is_weekend'] = df['weekday'].isin([0, 6]).astype(int)
df = df.drop(columns=['casual', 'registered'])
df[['hr', 'hour_sin', 'hour_cos', 'is_commute', 'is_weekend']].head()

## 6. Train and predict
I use a train-test split and a pipeline so scaling and encoding happen correctly inside the model workflow. The target is modeled as log rentals and then converted back to the original scale.

In [ ]:
feature_num = ['temp', 'atemp', 'hum', 'windspeed', 'hour_sin', 'hour_cos', 'is_commute', 'is_weekend']
feature_cat = ['season_name', 'weathersit_name']
X = df[feature_num + feature_cat]
y = df['cnt_log']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
numeric_transformer = Pipeline(steps=[('scaler', StandardScaler())])
categorical_transformer = Pipeline(steps=[('onehot', OneHotEncoder(handle_unknown='ignore'))])
preprocessor = ColumnTransformer(transformers=[('num', numeric_transformer, feature_num), ('cat', categorical_transformer, feature_cat)])
model = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', RandomForestRegressor(n_estimators=200, random_state=42))])
model.fit(X_train, y_train)
y_pred_log = model.predict(X_test)
y_pred = np.expm1(y_pred_log)
y_test_original = np.expm1(y_test)
predictions = pd.DataFrame({'actual_cnt': y_test_original, 'predicted_cnt': y_pred})
predictions.head()

## 7. Error calculation
I calculate MAE, RMSE, and R2, and I also keep the row-level prediction errors so the test set can be inspected directly.

In [ ]:
mae = mean_absolute_error(y_test_original, y_pred)
rmse = np.sqrt(mean_squared_error(y_test_original, y_pred))
r2 = r2_score(y_test_original, y_pred)
predictions['error'] = predictions['actual_cnt'] - predictions['predicted_cnt']
print('MAE:', round(mae, 2))
print('RMSE:', round(rmse, 2))
print('R2:', round(r2, 3))
display(predictions.head())
predictions.to_csv('test_predictions.csv', index=False)

## 8. Short conclusion
The model uses season, weather, and time-of-day patterns to predict hourly bike rentals, and the test metrics show how close the predictions are to the true rental counts.